Bronze Layer- Handling Raw Data

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
#Creating rating data 
ratings_data = [
    (1,101,4,1622388000000),
    (1,102,3,1622388020000),
    (2,101,5,1622388040000),
    (2,103,4,1622388060000),
    (3,101,3,1622388080000),
    (3,102,4,1622388100000),
    (3,103,None,1622388120000),
    (4,101,2,1622388140000)
]
columns_rating = ["user_id","movie_id","rating","timestamp"]
ratings_df = spark.createDataFrame(ratings_data, columns_rating)
ratings_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      3|     103|  NULL|1622388120000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
#Creating movies dataframe
movies_data = [
    (101,"The Last Kingdom","Action",2018),
    (102,"Love in Paris","Romance",2020),
    (103,"Galaxy Warriors","Sci-Fi",2019),
    (104,"Mystery Mansion","Thriller",2021),
    (105,"Laugh Out Loud","Comedy",2017),
    (106,"The Silent Forest","Drama",2016),
    (107,"Fast Track","Action",2022),
    (108,"Deep Ocean Secrets","Documentary",2015)
]
columns_movies = ["movie_id","movie_name","genre","release_year"]
movies_df = spark.createDataFrame(movies_data, columns_movies)
movies_df.show()

+--------+------------------+-----------+------------+
|movie_id|        movie_name|      genre|release_year|
+--------+------------------+-----------+------------+
|     101|  The Last Kingdom|     Action|        2018|
|     102|     Love in Paris|    Romance|        2020|
|     103|   Galaxy Warriors|     Sci-Fi|        2019|
|     104|   Mystery Mansion|   Thriller|        2021|
|     105|    Laugh Out Loud|     Comedy|        2017|
|     106| The Silent Forest|      Drama|        2016|
|     107|        Fast Track|     Action|        2022|
|     108|Deep Ocean Secrets|Documentary|        2015|
+--------+------------------+-----------+------------+



Silver Layer- Data Cleaning

In [0]:
#Handling null data 
from pyspark.sql.functions import col

clean_ratings_df = ratings_df.filter(col("rating").isNotNull())
display(clean_ratings_df)

user_id,movie_id,rating,timestamp
1,101,4,1622388000000
1,102,3,1622388020000
2,101,5,1622388040000
2,103,4,1622388060000
3,101,3,1622388080000
3,102,4,1622388100000
4,101,2,1622388140000


In [0]:
#Removing duplicates
clean_ratings_df = clean_ratings_df.dropDuplicates()
clean_ratings_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
#Validating rating range (1–5)
clean_ratings_df = clean_ratings_df.filter((col("rating") >= 1) & (col("rating") <= 5))

In [0]:
#Joining user rating table with movies table
joined_df = clean_ratings_df.join(movies_df, "movie_id", "inner")
joined_df.show()

+--------+-------+------+-------------+----------------+-------+------------+
|movie_id|user_id|rating|    timestamp|      movie_name|  genre|release_year|
+--------+-------+------+-------------+----------------+-------+------------+
|     101|      1|     4|1622388000000|The Last Kingdom| Action|        2018|
|     102|      1|     3|1622388020000|   Love in Paris|Romance|        2020|
|     101|      2|     5|1622388040000|The Last Kingdom| Action|        2018|
|     103|      2|     4|1622388060000| Galaxy Warriors| Sci-Fi|        2019|
|     101|      3|     3|1622388080000|The Last Kingdom| Action|        2018|
|     102|      3|     4|1622388100000|   Love in Paris|Romance|        2020|
|     101|      4|     2|1622388140000|The Last Kingdom| Action|        2018|
+--------+-------+------+-------------+----------------+-------+------------+



Gold Layer- Transformation and Aggregations

In [0]:
#Finding average rating for each movie
from pyspark.sql.functions import avg

avg_rating_df = joined_df.groupBy("movie_id","movie_name") \
    .agg(avg("rating").alias("avg_rating"))

avg_rating_df.show()

+--------+----------------+----------+
|movie_id|      movie_name|avg_rating|
+--------+----------------+----------+
|     101|The Last Kingdom|       3.5|
|     102|   Love in Paris|       3.5|
|     103| Galaxy Warriors|       4.0|
+--------+----------------+----------+



In [0]:
#Finding trending movies (high rating counts)
from pyspark.sql.functions import count

trending_df = joined_df.groupBy("movie_id","movie_name") \
    .agg(count("rating").alias("rating_count")) \
    .orderBy(col("rating_count").desc())

trending_df.show()

+--------+----------------+------------+
|movie_id|      movie_name|rating_count|
+--------+----------------+------------+
|     101|The Last Kingdom|           4|
|     102|   Love in Paris|           2|
|     103| Galaxy Warriors|           1|
+--------+----------------+------------+



In [0]:
#Finding low rating counts
low_quality_df = avg_rating_df.filter(col("avg_rating") < 3)
low_quality_df.show()

+--------+----------+----------+
|movie_id|movie_name|avg_rating|
+--------+----------+----------+
+--------+----------+----------+



In [0]:
#Finding Genre-wise Analytics
genre_df = joined_df.groupBy("genre") \
    .agg(avg("rating").alias("avg_rating"))

genre_df.show()

+-------+----------+
|  genre|avg_rating|
+-------+----------+
| Action|       3.5|
|Romance|       3.5|
| Sci-Fi|       4.0|
+-------+----------+



In [0]:
# Handling Incremental Data 
from pyspark.sql.functions import max

last_timestamp = ratings_df.agg(max("timestamp")).collect()[0][0]

# Example: filtering new data
incremental_df = ratings_df.filter(col("timestamp") > last_timestamp)

In [0]:
#Storing Data in Delta Lake
avg_rating_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/pipeline/delta/avg_ratings")

trending_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/pipeline/delta/trending_movies")

genre_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/pipeline/delta/genre_analysis")